In [45]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor

In [35]:
df = pd.read_csv("ml_dataset.csv")

print(df.shape)
print(df.head())

(17014, 126)
  delivery_day  hour  price_value  buy_volume_value  sell_volume_value  \
0   2024-10-01     0         3.31            4322.7             2079.3   
1   2024-10-01     1         0.08            4277.1             1849.0   
2   2024-10-01     2         0.05            4277.2             1938.7   
3   2024-10-01     3         0.02            4291.4             1928.2   
4   2024-10-01     4         8.00            4278.9             2149.6   

   net_volume  flow_DK1_export  flow_DK1_import  flow_GER_export  \
0      2243.4              0.0           1425.0              0.0   
1      2428.1              0.0           1425.0              0.0   
2      2338.5              0.0           1425.0              0.0   
3      2363.2              0.0           1425.0              0.0   
4      2129.3              0.0           1425.0              0.0   

   flow_GER_import  ...  price_BE_lag168  price_FR_lag168  \
0           1036.4  ...              NaN              NaN   
1          

In [36]:
print(df.columns)

Index(['delivery_day', 'hour', 'price_value', 'buy_volume_value',
       'sell_volume_value', 'net_volume', 'flow_DK1_export', 'flow_DK1_import',
       'flow_GER_export', 'flow_GER_import',
       ...
       'price_BE_lag168', 'price_FR_lag168',
       'no_2_forecasted_generation_lag168',
       'no_1_forecasted_generation_lag168',
       'no_5_forecasted_generation_lag168',
       'de_lu_forecasted_generation_lag168', 'no_2_forecasted_load_lag168',
       'no_1_forecasted_load_lag168', 'no_5_forecasted_load_lag168',
       'de_lu_forecasted_load_lag168'],
      dtype='str', length=126)


In [37]:
df["delivery_day"] = pd.to_datetime(df["delivery_day"])

#optionally filter out data before 2025-01-01
df = df[df["delivery_day"] >= "2024-01-01"]

# Target: price 24 hours ahead
df = df.sort_values(["delivery_day", "hour"]).reset_index(drop=True)

df["target_price_next_day"] = df["price_value"].shift(-24)

# Drop rows without a future price (last 24 rows)
df = df.dropna(subset=["target_price_next_day"])

print(df.shape)
df[["delivery_day", "hour", "price_value", "target_price_next_day"]].head(10)

(16990, 127)


C:\Users\torju\AppData\Local\Temp\ipykernel_34400\852458021.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target_price_next_day"] = df["price_value"].shift(-24)


,delivery_day,hour,price_value,target_price_next_day
0,2024-10-01,0,3.31,46.73
1,2024-10-01,1,0.08,47.84
2,2024-10-01,2,0.05,48.64
3,2024-10-01,3,0.02,47.57
4,2024-10-01,4,8.00,47.23
5,2024-10-01,5,25.25,47.32
6,2024-10-01,6,33.35,50.22
7,2024-10-01,7,38.03,81.12
8,2024-10-01,8,41.32,122.54
9,2024-10-01,9,41.51,91.30


In [57]:
drop_cols = ['delivery_day', 'total_exports', 'price_BE', 'no_1_wind_onshore', 'de_lu_forecasted_load_lag168', 'day_of_week', 'no_2_forecasted_load_lag24', 'price_NL_lag72', 'price_FR_lag168', 'flow_DK1_import', 'de_lu_forecasted_load_lag48', 'price_NL', 'day_sin', 'no_2_reservoir_level_MWh', 'price_BE_lag72', 'no_2_forecasted_load_lag168', 'price_NO1_lag24', 'price_NL_lag48', 'price_DE_LU_lag72', 'price_NO1_lag168', 'price_NO5_lag48', 'price_BE_lag24', 'no_2_forecasted_load_lag72', 'price_NO5_lag72', 'flow_NO1_export', 'no_2_forecasted_generation_lag72', 'price_lag_48', 'no_5_forecasted_generation_lag72', 'price_NL_lag168', 'price_NO5_lag168', 'year_cos', 'no_5_reservoir_level_MWh', 'no_5_forecasted_load_lag24', 'no_1_forecasted_generation_lag48', 'net_volume', 'no_5_forecasted_generation_lag24', 'no_5_forecasted_load', 'de_lu_wind_offshore', 'price_DK1_lag72', 'no_5_forecasted_generation', 'no_5_forecasted_load_lag48', 'price_NO1_lag72', 'no_5_forecasted_generation_lag48', 'no_1_forecasted_load_lag72', 'no_2_forecasted_load', 'no_1_forecasted_load_lag24', 'no_5_forecasted_load_lag72', 'flow_DK1_export', 'de_lu_forecasted_generation', 'price_BE_lag48', 'no_2_forecasted_generation_lag168', 'no_1_reservoir_level_MWh', 'buy_volume_value', 'price_DK1', 'price_FR', 'no_2_wind_offshore', 'no_2_wind_onshore', 'no_5_forecasted_generation_lag168', 'de_lu_forecasted_generation_lag168', 'day_cos', 'price_FR_lag24', 'de_lu_solar', 'de_lu_wind_onshore', 'no_1_forecasted_generation_lag24', 'price_lag_168', 'no_1_forecasted_load', 'de_lu_forecasted_load', 'price_NO5_lag24', 'price_DK1_lag24', 'year_sin', 'no_2_forecasted_generation_lag48', 'de_lu_forecasted_load_lag72', 'price_DE_LU_lag48', 'price_lag_72', 'no_2_forecasted_generation_lag24', 'no_1_forecasted_generation_lag168', 'de_lu_forecasted_load_lag24', 'de_lu_forecasted_generation_lag72', 'price_FR_lag48', 'price_DE_LU_lag168', 'de_lu_forecasted_generation_lag24', 'capacity_NO5', 'no_1_forecasted_generation_lag72', 'flow_NO5_import', 'price_DE_LU_lag24', 'no_2_forecasted_generation', 'net_flow', 'price_DK1_lag48', 'no_1_forecasted_load_lag48', 'total_imports', 'sell_volume_value', 'flow_GER_export', 'no_1_forecasted_generation', 'month', 'price_NL_lag24', 'hour', 'flow_NO1_import', 'no_2_forecasted_load_lag48', 'flow_NL_export', 'flow_GER_import', 'flow_NO5_export', 'flow_NL_import', 'no_5_wind_onshore', 'no_1_solar', 'no_5_solar', 'capacity_NO1', 'util_NL', 'capacity_NL', 'no_2_solar', 'capacity_GER', 'util_GER', 'util_NO1', 'util_NO5']
drop_cols = ['delivery_day']

X = df.drop(columns=drop_cols + ["target_price_next_day"])
y = df["target_price_next_day"]

print("Features:", len(X.columns))

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

Features: 125
Train rows: 13,592
Test rows : 3,398


In [55]:
model = XGBRegressor(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.01,
        objective="reg:squarederror"
    )

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

Training...
Predicting...
done


In [56]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")
print("MAE:", mean_absolute_error(y_test, pred))

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_lag_24"]
)
print(f"price_lag_24 RMSE: {rmse:.4f}")

RMSE = 32.6681
MAE: 25.598274774714174
R²: 0.39121659555586297
price_lag_24 RMSE: 37.8668


In [49]:
importance = (
pd.DataFrame({
"feature": X.columns,
"importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
)
print(importance[importance["importance"] < 0.01]["feature"].tolist())
print(importance[importance["importance"] > 0.01])

['total_exports', 'price_BE', 'no_1_wind_onshore', 'de_lu_forecasted_load_lag168', 'day_of_week', 'no_2_forecasted_load_lag24', 'price_NL_lag72', 'price_FR_lag168', 'flow_DK1_import', 'de_lu_forecasted_load_lag48', 'price_NL', 'day_sin', 'no_2_reservoir_level_MWh', 'price_BE_lag72', 'no_2_forecasted_load_lag168', 'price_NO1_lag24', 'price_NL_lag48', 'price_DE_LU_lag72', 'price_NO1_lag168', 'price_NO5_lag48', 'price_BE_lag24', 'no_2_forecasted_load_lag72', 'price_NO5_lag72', 'flow_NO1_export', 'no_2_forecasted_generation_lag72', 'price_lag_48', 'no_5_forecasted_generation_lag72', 'price_NL_lag168', 'price_NO5_lag168', 'year_cos', 'no_5_reservoir_level_MWh', 'no_5_forecasted_load_lag24', 'no_1_forecasted_generation_lag48', 'net_volume', 'no_5_forecasted_generation_lag24', 'no_5_forecasted_load', 'de_lu_wind_offshore', 'price_DK1_lag72', 'no_5_forecasted_generation', 'no_5_forecasted_load_lag48', 'price_NO1_lag72', 'no_5_forecasted_generation_lag48', 'no_1_forecasted_load_lag72', 'no_2_fo